<a href="https://colab.research.google.com/github/suniltejh361-arch/dataviz-exercises-suniltejh361/blob/main/lecture05_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('/content/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


In [3]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

> 💡 `df_cap[df_cap['room_type'].isin(['Entire home/apt','Private room'])]`


In [4]:
# Task 1 — Histogram

import plotly.express as px


df_hist = df_cap[df_cap['room_type'].isin(['Entire home/apt', 'Private room'])]

median_entire = df_hist[df_hist['room_type']=='Entire home/apt']['price'].median()
median_private = df_hist[df_hist['room_type']=='Private room']['price'].median()

fig = px.histogram(
    df_hist,
    x='price',
    color='room_type',
    barmode='overlay',
    opacity=0.6,
    title='Entire homes are right-skewed and significantly more expensive than private rooms'
)


fig.add_vline(
    x=median_entire,
    line_dash="dash",
    line_color="blue",
    annotation_text=f"Median Entire: £{median_entire:.0f}",
    annotation_position="top left"
)

fig.add_vline(
    x=median_private,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Median Private: £{median_private:.0f}",
    annotation_position="top right"
)

fig.show()

## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out with before plotting

In [5]:
# Task 2 — Box Plot


df_box = df_cap[df_cap['reviews_per_month'] > 0]


median_reviews = df_box.groupby('neighbourhood')['reviews_per_month'].median().sort_values(ascending=False)

df_box['neighbourhood'] = pd.Categorical(
    df_box['neighbourhood'],
    categories=median_reviews.index,
    ordered=True
)


top2 = median_reviews.head(2).index.tolist()

df_box['highlight'] = df_box['neighbourhood'].apply(
    lambda x: 'Top Boroughs' if x in top2 else 'Other Boroughs'
)

fig = px.box(
    df_box,
    x='reviews_per_month',
    y='neighbourhood',
    color='highlight',
    points='outliers',
    title=f"{top2[0]} and {top2[1]} are the most active boroughs based on booking frequency"
)

fig.show()

/tmp/ipykernel_3132/2381498560.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_3132/2381498560.py:20: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

